# Week 5 - LSTM
Sentence predictions


In [1]:
# !wget -O great_gatsby.txt "https://www.gutenberg.org/cache/epub/64317/pg64317.txt"

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
#from torchtext.vocab import build_vocab_from_iterator
#import spacy
import numpy as np
import random
import math
import time

In [3]:
SEED = 1234
# Setting a seed ensures reproducibility of results in random processes.
# By setting the seed to a fixed value, random number generation becomes deterministic,
# meaning that the same sequence of random numbers will be generated each time the code is run.

# Set the seed for Python's built-in random module
random.seed(SEED)

# Set the seed for NumPy's random number generator
np.random.seed(SEED)

# Set the seed for PyTorch's random number generator on CPU
torch.manual_seed(SEED)

# Check if CUDA (GPU acceleration) is available on the system, or mps
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')


In [4]:
class LSTMLayer(nn.Module):
    def __init__(self, input_size, hidden_size):
        '''
          - Args:
                - input_size: The number of expected features in the input.
                - hidden_size: The number of features in the hidden state.
          - Functionality:
                - Initialises the LSTM layer with the specified input size, and hidden size.
                - Initialises weight and bias parameters.
        '''
        super(LSTMLayer, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size

        # Define parameters for a single LSTM layer
        # We mark the weight matrices as a parameter of the model using nn.Parameter
        # This is done to indicate that this tensor should be considered a model parameter.
        # So that backward function in pytorch consider these matrices during optimisation,
        # and to calculate the gradients with respect to them.
        self.W_f = nn.Parameter(torch.Tensor(input_size, hidden_size))
        self.U_f = nn.Parameter(torch.Tensor(hidden_size, hidden_size))
        self.b_f = nn.Parameter(torch.Tensor(hidden_size))

        self.W_i = nn.Parameter(torch.Tensor(input_size, hidden_size))
        self.U_i = nn.Parameter(torch.Tensor(hidden_size, hidden_size))
        self.b_i = nn.Parameter(torch.Tensor(hidden_size))

        self.W_g = nn.Parameter(torch.Tensor(input_size, hidden_size))
        self.U_g = nn.Parameter(torch.Tensor(hidden_size, hidden_size))
        self.b_g = nn.Parameter(torch.Tensor(hidden_size))

        self.W_o = nn.Parameter(torch.Tensor(input_size, hidden_size))
        self.U_o = nn.Parameter(torch.Tensor(hidden_size, hidden_size))
        self.b_o = nn.Parameter(torch.Tensor(hidden_size))

    def forward(self, input, h_prev, c_prev):
        # Concatenate input and previous hidden state

        # Forget gate
        # Now that you're becoming comfortable with matrix multiplication,
        # we have commented on what's happening with the sizes for only one equation.
        # We want you to comment on the other equations as well.
        # input: (batch_size, input_size) @ W_f: (input_size, hidden_size) -> (batch_size, hidden_size)
        # h_prev: (batch_size, hidden_size) @ U_f: (hidden_size, hidden_size) -> (batch_size, hidden_size)

        # input: (batch_size, input_size) @ W_f: (input_size, hidden_size) +
        # h_prev: (batch_size, hidden_size) @ U_f (hidden_size, hidden_size) +
        # b_f(hidden_size, ) -> (batch_size, hidden_size)
        f = torch.sigmoid(input @ self.W_f + h_prev @ self.U_f + self.b_f)

        # Element-wise multiplication requires same shape
        # f: (batch_size, hidden_size) * c_prev: (batch_size, hidden_size) -> (batch_size, hidden_size)
        k = f * c_prev #mask


        # Input gate (Add gate)
        # input: (batch_size, input_size) @ W_i: (input_size, hidden_size) +
        # h_prev: (batch_size, hidden_size) @ U_i: (hidden_size, hidden_size) +
        # b_i: (hidden_size, ) -> (batch_size, hidden_size)
        i = torch.sigmoid(input @ self.W_i + h_prev @ self.U_i + self.b_i)

        # input: (batch_size, input_size) @ W_g:(input_size, hidden_size) +
        # h_prev: (batch_size, hidden_size) @ U_g: (hidden_size, hidden_size) +
        # b_g: (hidden_size, ) -> (batch_size, hidden_size)
        g = torch.tanh(input @ self.W_g + h_prev @ self.U_g + self.b_g) # Candidate cell state

        # g: (batch_size, hidden_size) * i: (batch_size, hidden_size) -> (batch_size, hidden_size)
        j =  g * i #mask


        # input: (batch_size, input_size) @ W_o: (input_size, hidden_size) +
        # h_prev: (batch_size, hidden_size) @ U_o: (hidden_size, hidden_size) +
        # b_o: (hidden_size) -> (batch_size, hidden_size)
        # Output gate
        o = torch.sigmoid(input @ self.W_o + h_prev @ self.U_o + self.b_o)


        # Update cell state
        # j: (batch_size, hidden_size) + k: (batch_size, hidden_size) -> (batch_size, hidden_size)
        c_next = j + k

        # Update hidden state
        # o: (batch_size, hidden_size) * tanh(c_next): (batch_size, hidden_size) -> (batch_size, hidden_size)
        h_next = o * torch.tanh(c_next)

        return h_next, c_next

In [5]:
class StackLSTMLayers(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1):
        '''
          - Args:
                - input_size: The number of expected features in the input.
                - hidden_size: The number of features in the hidden state.
                - num_layers: Number of Stacked recurrent layers.
          - Functionality:
                - Initialises the LSTM layer with the specified input size, hidden size, and number of layers.
                - Initialises weight and bias parameters for each layer.
        '''
        super(StackLSTMLayers, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Create stack of LSTM layers if num_layers > 1
        self.layers = nn.ModuleList([LSTMLayer(input_size if i == 0 else hidden_size, hidden_size) for i in range(num_layers)])

    def forward(self, input, hidden=None):
        if hidden is None:
            # initialise hidden and cell states if not provided
            hidden = self.init_hidden(input.size(1))

        # Unpack hidden states
        hiddens, cells = hidden

        outputs = []

        # Iterate through each time step
        for input_t in input:
            # Iterate through each layer
            for layer_idx, layer in enumerate(self.layers):
                # Pass input through the current layer
                hiddens[layer_idx], cells[layer_idx] = layer(input_t, hiddens[layer_idx], cells[layer_idx])

                # Update input for the next layer (if any)
                if layer_idx < self.num_layers - 1:
                    input_t = hiddens[layer_idx]

            # Append output of current time step
            outputs.append(hiddens[-1])

        # Stack outputs along the sequence dimension
        outputs = torch.stack(outputs, dim=0)


        # Return outputs, hidden states, and cell states
        return outputs, (hiddens, cells)

    def init_hidden(self, batch_size):
        # initialise hidden and cell states for each layer
        hiddens = [torch.zeros(batch_size, self.hidden_size, device=device) for _ in range(self.num_layers)]
        cells = [torch.zeros(batch_size, self.hidden_size, device=device) for _ in range(self.num_layers)]
        return hiddens, cells

In [6]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()

        self.hid_dim = hid_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(input_dim, emb_dim)

        self.rnn = StackLSTMLayers(emb_dim, hid_dim, n_layers)

        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        #src = [src len, batch size]

        embedded = self.dropout(self.embedding(src)) ## UNDERSTAND THIS

        #embedded = [src len, batch size, emb dim]

        outputs, (hidden, cell) = self.rnn(embedded)

        #outputs = [src len, batch size, hid dim * n directions]
        #hidden = [n layers * n directions, batch size, hid dim]
        #cell = [n layers * n directions, batch size, hid dim]

        #outputs are always from the top hidden layer
        return hidden, cell

In [7]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):

        super().__init__()

        self.output_dim = output_dim
        self.hid_dim = hid_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(output_dim, emb_dim)

        self.rnn = StackLSTMLayers(emb_dim, hid_dim, n_layers)

        self.fc_out = nn.Linear(hid_dim, output_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):

        #input = [batch size]
        #hidden = [n layers * n directions, batch size, hid dim]
        #cell = [n layers * n directions, batch size, hid dim]

        #n directions in the decoder will both always be 1, therefore:
        #hidden = [n layers, batch size, hid dim]
        #context = [n layers, batch size, hid dim]

        input = input.unsqueeze(0)

        #input = [1, batch size]

        embedded = self.dropout(self.embedding(input))

        #embedded = [1, batch size, emb dim]

        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))

        #output = [seq len, batch size, hid dim * n directions]
        #hidden = [n layers * n directions, batch size, hid dim]
        #cell = [n layers * n directions, batch size, hid dim]

        #seq len and n directions will always be 1 in the decoder, therefore:
        #output = [1, batch size, hid dim]
        #hidden = [n layers, batch size, hid dim]
        #cell = [n layers, batch size, hid dim]

        prediction = self.fc_out(output.squeeze(0))

        #prediction = [batch size, output dim]

        return prediction, hidden, cell

In [8]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device

        assert encoder.hid_dim == decoder.hid_dim, \
            "Hidden dimensions of encoder and decoder must be equal!"
        assert encoder.n_layers == decoder.n_layers, \
            "Encoder and decoder must have equal number of layers!"

    def forward(self, src, trg, teacher_forcing_ratio = 0.5):

        #src = [src len, batch size]
        #trg = [trg len, batch size]
        #teacher_forcing_ratio is probability to use teacher forcing
        #e.g. if teacher_forcing_ratio is 0.75 we use ground-truth inputs 75% of the time

        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        #tensor to store decoder outputs
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)

        #last hidden state of the encoder is used as the initial hidden state of the decoder
        hidden, cell = self.encoder(src)

        #first input to the decoder is the <sos> tokens
        input = trg[0,:]

        for t in range(1, trg_len):

            #insert input token embedding, previous hidden and previous cell states
            #receive output tensor (predictions) and new hidden and cell states
            output, hidden, cell = self.decoder(input, hidden, cell)

            #place predictions in a tensor holding predictions for each token
            outputs[t] = output

            #decide if we are going to use teacher forcing or not
            teacher_force = random.random() < teacher_forcing_ratio

            #get the highest predicted token from our predictions
            top1 = output.argmax(1)

            #if teacher forcing, use actual next token as next input
            #if not, use predicted token
            input = trg[t] if teacher_force else top1

        return outputs

In [9]:
def train(model, iterator, optimizer, criterion, clip):

    model.train()

    epoch_loss = 0

    num_batches = 0
    dataloader = DataLoader(iterator, batch_size=BATCH_SIZE, collate_fn=collate_fn)

    for src, trg in dataloader:

        src = src.to(device)
        trg = trg.to(device)

        optimizer.zero_grad()

        output = model(src, trg)

        #trg = [trg len, batch size]
        #output = [trg len, batch size, output dim]

        output_dim = output.shape[-1]

        output = output[1:].view(-1, output_dim)
        trg = trg[1:].view(-1)

        #trg = [(trg len - 1) * batch size]
        #output = [(trg len - 1) * batch size, output dim]

        loss = criterion(output, trg)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    return epoch_loss / max(1, num_batches)

### Load and Clean the Text

In [10]:
import spacy
# Read the file as a string
with open("great_gatsby.txt", "r", encoding="utf-8") as f:
    book_as_string = f.read()

# Strip Gutenberg header/footer
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK THE GREAT GATSBY ***"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK THE GREAT GATSBY ***"

start_idx = book_as_string.find(start_marker) + len(start_marker)
end_idx = book_as_string.find(end_marker)

clean_text = book_as_string[start_idx:end_idx].strip()

# Split into sentences
nlp = spacy.load("en_core_web_sm")
doc = nlp(clean_text)
sentences = [sent.text.strip() for sent in doc.sents]

### Tokenize and Build ONE Vocabulary
Need to understand this part better

In [11]:
from torchtext.vocab import build_vocab_from_iterator
def tokenize(text):
    return [tok.text.lower() for tok in nlp.tokenizer(text)]

# Yield tokens from sentences
def yield_tokens(sentences):
    for sentence in sentences:
        yield tokenize(sentence)

# Define special symbols and indices
UNK_IDX, PAD_IDX, BOS_IDX, EOS_IDX = 0, 1, 2, 3
special_symbols = ['<unk>', '<pad>', '<bos>', '<eos>']

# Build ONE vocabulary from all sentences
vocab = build_vocab_from_iterator(yield_tokens(sentences),
                                  min_freq=2,
                                  specials=special_symbols,
                                  special_first=True)

vocab.set_default_index(UNK_IDX)

### Create a Dataset Class with Shifted Pairs

In [12]:
class LanguageModelDataset(Dataset):
    """
    Dataset for language modeling with shifted pairs.

    For each sentence [w1, w2, ..., wN], returns:
        src (input):  [<bos>, w1, w2, ..., wN]  - what the model reads
        trg (target): [w1, w2, ..., wN, <eos>]  - what the model predicts

    The target is the input shifted by one token.
    """

    def __init__(self, sentences, vocab, tokenize_fn):
        self.sentences = sentences
        self.vocab = vocab
        self.tokenize_fn = tokenize_fn

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sentence = self.sentences[idx]
        tokens = self.tokenize_fn(sentence)
        token_indices = self.vocab(tokens)  # Convert tokens to indices

        # Create shifted pairs
        src = [BOS_IDX] + token_indices          # [<bos>, w1, w2, ..., wN]
        trg = token_indices + [EOS_IDX]          # [w1, w2, ..., wN, <eos>]

        return torch.tensor(src, dtype=torch.long), torch.tensor(trg, dtype=torch.long)

### Split into Train/Validation/Test

In [13]:
random.shuffle(sentences)
n = len(sentences)
train_sents = sentences[:int(0.8*n)]
valid_sents = sentences[int(0.8*n):int(0.9*n)]
test_sents  = sentences[int(0.9*n):]

### LanguageModelDataset objects

In [14]:
train_dataset = LanguageModelDataset(train_sents, vocab, tokenize)
valid_dataset = LanguageModelDataset(valid_sents, vocab, tokenize)
test_dataset = LanguageModelDataset(test_sents, vocab, tokenize)

### Collate Function

In [15]:
def collate_fn(batch):
    src_batch, trg_batch = [], []
    for src, trg in batch:
        src_batch.append(src)
        trg_batch.append(trg)
    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX)
    trg_batch = pad_sequence(trg_batch, padding_value=PAD_IDX)
    return src_batch, trg_batch

### Model Initialization

In [16]:
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 2
ENC_DROPOUT = 0.2
DEC_DROPOUT = 0.2
BATCH_SIZE = 64

VOCAB_SIZE = len(vocab)  # single vocabulary
enc = Encoder(VOCAB_SIZE, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
dec = Decoder(VOCAB_SIZE, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)
model = Seq2Seq(enc, dec, device).to(device)

def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)

model.apply(init_weights)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'The model has {count_parameters(model):,} trainable parameters')

The model has 10,181,324 trainable parameters


In [17]:
optimizer = optim.Adam(model.parameters())

In [18]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

In [19]:
def train(model, iterator, optimizer, criterion, clip):

    model.train()

    epoch_loss = 0

    num_batches = 0
    dataloader = DataLoader(iterator, batch_size=BATCH_SIZE, collate_fn=collate_fn)

    for src, trg in dataloader:

        src = src.to(device)
        trg = trg.to(device)

        optimizer.zero_grad()

        output = model(src, trg)

        #trg = [trg len, batch size]
        #output = [trg len, batch size, output dim]

        output_dim = output.shape[-1]

        output = output[1:].view(-1, output_dim)
        trg = trg[1:].view(-1)

        #trg = [(trg len - 1) * batch size]
        #output = [(trg len - 1) * batch size, output dim]

        loss = criterion(output, trg)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    return epoch_loss / max(1, num_batches)

In [20]:
def evaluate(model, iterator, criterion):

    model.eval()

    epoch_loss = 0

    num_batches = 0
    dataloader = DataLoader(iterator, batch_size=BATCH_SIZE, collate_fn=collate_fn)

    with torch.no_grad():

        for src, trg in dataloader:
            src = src.to(device)
            trg = trg.to(device)

            output = model(src, trg, 0) #turn off teacher forcing

            #trg = [trg len, batch size]
            #output = [trg len, batch size, output dim]

            output_dim = output.shape[-1]

            output = output[1:].view(-1, output_dim)
            trg = trg[1:].view(-1)

            #trg = [(trg len - 1) * batch size]
            #output = [(trg len - 1) * batch size, output dim]

            loss = criterion(output, trg)

            epoch_loss += loss.item()
            num_batches += 1

    return epoch_loss / max(1, num_batches)

In [21]:
def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

In [22]:
N_EPOCHS = 30
CLIP = 1

best_valid_loss = float('inf')

for epoch in range(N_EPOCHS):

    start_time = time.time()

    train_loss = train(model, train_dataset, optimizer, criterion, CLIP)
    valid_loss = evaluate(model, valid_dataset, criterion)

    end_time = time.time()

    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'lstm-seq2seq-model.pt')

    print(f'Epoch: {epoch+1:02} | Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {math.exp(valid_loss):7.3f}')

Epoch: 01 | Time: 0m 24s
	Train Loss: 5.953 | Train PPL: 385.072
	 Val. Loss: 5.589 |  Val. PPL: 267.548
Epoch: 02 | Time: 0m 21s
	Train Loss: 5.553 | Train PPL: 258.061
	 Val. Loss: 5.562 |  Val. PPL: 260.446
Epoch: 03 | Time: 0m 21s
	Train Loss: 5.484 | Train PPL: 240.828
	 Val. Loss: 5.573 |  Val. PPL: 263.349
Epoch: 04 | Time: 0m 21s
	Train Loss: 5.427 | Train PPL: 227.482
	 Val. Loss: 5.561 |  Val. PPL: 260.097
Epoch: 05 | Time: 0m 21s
	Train Loss: 5.355 | Train PPL: 211.723
	 Val. Loss: 5.540 |  Val. PPL: 254.758
Epoch: 06 | Time: 0m 21s
	Train Loss: 5.310 | Train PPL: 202.383
	 Val. Loss: 5.566 |  Val. PPL: 261.449
Epoch: 07 | Time: 0m 21s
	Train Loss: 5.215 | Train PPL: 184.098
	 Val. Loss: 5.617 |  Val. PPL: 275.140
Epoch: 08 | Time: 0m 22s
	Train Loss: 5.138 | Train PPL: 170.445
	 Val. Loss: 5.583 |  Val. PPL: 265.988


KeyboardInterrupt: 

In [ ]:
model.load_state_dict(torch.load('lstm-seq2seq-model.pt'))

loss = evaluate(model, valid_dataset, criterion)

print(f'| Loss: {loss:.3f} | PPL: {math.exp(loss):7.3f} |')

| Loss: 5.251 | PPL: 190.852 |


### Step 10: Inference
1. Pick 10 sentences from test set
2. For each, tokenize and convert to indices to create the src tensor
3. Run the encoder to get the context vector
4. Run the decoder token-by-token (with teacher_forcing_ratio=0) to generate predictions
5. Convert the predicted indices back to words using the vocabulary
6. Print the input sentence and the model's predicted continuation

In [ ]:
# Build reverse vocabulary mapping (index -> token)
itos = vocab.get_itos()

model.eval()

# Pick 10 test sentences with at least 5 tokens (to get meaningful predictions)
test_samples = [s for s in test_sents if len(tokenize(s)) >= 5][:10]

print("=" * 80)
print("INFERENCE: 10 Test Inputs from Moby Dick")
print("=" * 80)

with torch.no_grad():
    for i, sentence in enumerate(test_samples):
        # Tokenize and convert to vocabulary indices
        tokens = tokenize(sentence)
        token_indices = vocab(tokens)

        # Create src and trg tensors matching the dataset format
        # src: [<bos>, w1, w2, ..., wN]   shape: [src_len, 1]
        # trg: [w1, w2, ..., wN, <eos>]   shape: [trg_len, 1]
        src_tensor = torch.tensor(
            [BOS_IDX] + token_indices, dtype=torch.long
        ).unsqueeze(1).to(device)

        trg_tensor = torch.tensor(
            token_indices + [EOS_IDX], dtype=torch.long
        ).unsqueeze(1).to(device)

        # Forward pass with teacher forcing OFF (model uses only its own predictions)
        output = model(src_tensor, trg_tensor, teacher_forcing_ratio=0)

        # output shape: [trg_len, 1, vocab_size]
        # Position 0 is zeros (decoder loop starts at t=1)
        # Positions 1..N contain predictions for w2, w3, ..., wN, <eos>
        predicted_indices = output[1:].argmax(2).squeeze(1).cpu().tolist()
        predicted_tokens = [itos[idx] for idx in predicted_indices]

        # Ground truth target for comparison
        target_tokens = tokens[1:] + ["<eos>"]

        print(f"\nTest {i+1}:")
        print(f"  Input:     {sentence}")
        print(f"  Target:    {' '.join(target_tokens)}")
        print(f"  Predicted: {' '.join(predicted_tokens)}")

INFERENCE: 10 Test Inputs from Moby Dick

Test 1:
  Input:     VI

About this time an ambitious young reporter from New York arrived one
morning at Gatsby’s door and asked him if he had anything to say.
  Target:    

 about this time an ambitious young reporter from new york arrived one 
 morning at gatsby ’s door and asked him if he had anything to say . <eos>
  Predicted: <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> . <eos> <eos> <eos>

Test 2:
  Input:     and I said: ‘All right, Katspaugh,
don’t pay him a penny till he shuts his mouth.’
  Target:    i said : ‘ all right , katspaugh , 
 do n’t pay him a penny till he shuts his mouth . ’ <eos>
  Predicted: i did n’t be 
 
 , 
 , and 
 <unk> , and 
 <unk> , and <unk> <unk> <unk> . <eos> <eos>

Test 3:
  Input:     A souvenir of Oxford days.
  Target:    souvenir of oxford days . <eos>
  Predicted: <unk> <unk> <unk> <unk> . <